# Document ingestion: PDF and Word (.docx)

Goal: attach a PDF and a `.docx` file to an offline agent run and see
exactly what the model receives. Also exercise the debug parse helpers
and PDF classification directly, without an `Agent` or `Run`.

The document-ingestion extension is two crates working together:

- `finstack-ai-tools-document` converts pdf/docx/doc/xlsx/xls/pptx/ppt/
  odt/ods/odp/rtf/epub/csv to GitHub-Flavored Markdown (`document_parse`)
  and classifies a PDF's page content as `text`/`scanned`/`mixed`/`image`
  (`pdf_classify`) — no OCR is performed.
- `finstack-ai-middleware-document-ingest` is a fail-soft `BeforeModel`
  middleware that rewrites an attached `File` block into model-visible
  Markdown before the request reaches the model. The canonical,
  journaled conversation keeps the original `File` block unchanged; only
  the model-visible request is rewritten.

Trust: [T2](../../docs/site/security-trust-levels.md) trusted
callback. Network: none. Mirrors
`bindings/finstack-ai-python/tests/test_document_attachments.py` and
`bindings/finstack-ai-python/tests/test_debug_parse_document.py`.

## 1. Quick debug parsing (no agent, no run)

`finstack_ai.parse_document_markdown` and `finstack_ai.parse_document`
call the toolset's `parser::parse` directly. They exist so a developer
can see exactly what Markdown the ingest middleware would inject for a
given file, without constructing an `Agent`. Both take `media_type` plus
exactly one of `data=` (in-memory bytes) or `path=` (read at call time,
bounded to 4 MiB).

Fixture paths below are relative to this notebook's location
(`examples/python-notebooks/`), reaching up into the shared
`fixtures/documents/` corpus used by the Rust crate's own tests.

In [1]:
from pathlib import Path

import finstack_ai

FIXTURES = Path("../../fixtures/documents").resolve()
assert FIXTURES.is_dir(), FIXTURES

PDF_MEDIA_TYPE = "application/pdf"
DOCX_MEDIA_TYPE = (
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
)

### 1a. PDF: markdown-only helper, then the detailed result

In [2]:
pdf_bytes = (FIXTURES / "text.pdf").read_bytes()

pdf_markdown = finstack_ai.parse_document_markdown(
    media_type=PDF_MEDIA_TYPE, data=pdf_bytes
)
print(pdf_markdown)

## Quarterly Revenue Report



In [3]:
pdf_detail = finstack_ai.parse_document(media_type=PDF_MEDIA_TYPE, data=pdf_bytes)
pdf_detail

{'markdown': '## Quarterly Revenue Report\n',
 'format': 'pdf',
 'page_count': 1,
 'classification': 'text',
 'requires_ocr': False,
 'truncated': False}

`parse_document` returns the full `ParsedDocument`: `markdown`,
`format`, `page_count`, `classification`, `requires_ocr`, `truncated`.
For a normal text PDF, `classification` is `"text"` and `requires_ocr`
is `False`.

### 1b. Word (.docx): the same two calls

A `.docx` is a container format (via `anydoc`), not a PDF, so
`page_count`/`classification`/`requires_ocr` stay `None`/`None`/`False`
— those fields are PDF-specific.

In [4]:
docx_bytes = (FIXTURES / "sample.docx").read_bytes()

docx_markdown = finstack_ai.parse_document_markdown(
    media_type=DOCX_MEDIA_TYPE, data=docx_bytes
)
print(docx_markdown)

Hello from a docx fixture.



In [5]:
docx_detail = finstack_ai.parse_document(media_type=DOCX_MEDIA_TYPE, data=docx_bytes)
docx_detail

{'markdown': 'Hello from a docx fixture.\n',
 'format': 'docx',
 'page_count': None,
 'classification': None,
 'requires_ocr': False,
 'truncated': False}

## 2. PDF classification: a scanned PDF

A scanned or image-only PDF is a **successful** parse, not an error:
`requires_ocr` is `True` and `markdown` may be empty. Callers must check
`requires_ocr` rather than treating empty `markdown` as failure.
`pdf_classify` (used by the ingest middleware; not called directly
here) exposes the same signal ahead of time via `classification`.

In [6]:
scanned_bytes = (FIXTURES / "scanned.pdf").read_bytes()

scanned_detail = finstack_ai.parse_document(
    media_type=PDF_MEDIA_TYPE, data=scanned_bytes
)
scanned_detail

{'markdown': '',
 'format': 'pdf',
 'page_count': 1,
 'classification': 'scanned',
 'requires_ocr': True,
 'truncated': False}

In [7]:
assert scanned_detail["requires_ocr"] is True
assert scanned_detail["classification"] == "scanned"
assert scanned_detail["markdown"] == ""
print("scanned PDF confirmed: requires_ocr=True, markdown empty")

scanned PDF confirmed: requires_ocr=True, markdown empty


## 3. Agent run with a PDF attachment

Now attach the PDF to a real (offline, scripted) agent run and inspect
the model-visible request. `finstack_ai.PythonModel` wraps a Python
callback as the model; `finstack_ai.Agent.from_python` builds an agent
around it entirely in-process, no network and no API key.

The callback below captures the exact `ModelRequestDraft` JSON the Rust
engine hands it — the same shape
`bindings/finstack-ai-python/src/callbacks/model.rs::PythonModelAdapter::request`
serializes — so we can inspect the model-visible message content after
`DocumentIngestMiddleware`'s `BeforeModel` rewrite has already run.

In [8]:
from typing import Any

captured_requests: list[dict[str, Any]] = []


async def capturing_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context
    captured_requests.append(request)
    return {"text": "acknowledged", "completion_id": "notebook-pdf-1"}


model = finstack_ai.PythonModel(
    capturing_model,
    component="notebook.model.document-ingestion",
    provider="notebook",
    model="notebook-model",
)
agent = await finstack_ai.Agent.from_python(model)

In [9]:
pdf_attachment = finstack_ai.Attachment(
    media_type=PDF_MEDIA_TYPE, path=str(FIXTURES / "text.pdf")
)

result = await agent.run(
    "Summarize the attached file.", attachments=[pdf_attachment]
)
print(result.text)

acknowledged


Pull apart the request the model actually saw. The middleware replaces
the `File` block with `Text` — the model never sees a raw PDF.

In [10]:
request = captured_requests[-1]
user_messages = [m for m in request["messages"] if m["role"] == "user"]
user_blocks = [block for message in user_messages for block in message["content"]]

print("block kinds:", [block["kind"] for block in user_blocks])
assert not any(block["kind"] == "file" for block in user_blocks), (
    "model must never see a raw File block"
)

injected_markdown = "".join(
    block["text"] for block in user_blocks if block["kind"] == "text"
)
print(injected_markdown)

block kinds: ['text', 'text']
Summarize the attached file.Attached document "text.pdf" (Pdf, 1 pages), converted to Markdown:

## Quarterly Revenue Report



## 4. Agent run with a Word (.docx) attachment

Same pattern, a different document format. A fresh capture list keeps
this run's request separate from the PDF run above.

In [11]:
docx_captured_requests: list[dict[str, Any]] = []


async def docx_capturing_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context
    docx_captured_requests.append(request)
    return {"text": "acknowledged", "completion_id": "notebook-docx-1"}


docx_model = finstack_ai.PythonModel(
    docx_capturing_model,
    component="notebook.model.document-ingestion-docx",
    provider="notebook",
    model="notebook-model",
)
docx_agent = await finstack_ai.Agent.from_python(docx_model)

docx_attachment = finstack_ai.Attachment(
    media_type=DOCX_MEDIA_TYPE, path=str(FIXTURES / "sample.docx")
)

docx_result = await docx_agent.run(
    "Summarize the attached file.", attachments=[docx_attachment]
)
print(docx_result.text)

acknowledged


In [12]:
docx_request = docx_captured_requests[-1]
docx_user_messages = [
    m for m in docx_request["messages"] if m["role"] == "user"
]
docx_user_blocks = [
    block for message in docx_user_messages for block in message["content"]
]

print("block kinds:", [block["kind"] for block in docx_user_blocks])
assert not any(block["kind"] == "file" for block in docx_user_blocks), (
    "model must never see a raw File block"
)

docx_injected_markdown = "".join(
    block["text"] for block in docx_user_blocks if block["kind"] == "text"
)
print(docx_injected_markdown)

block kinds: ['text', 'text']
Summarize the attached file.Attached document "sample.docx" (Docx), converted to Markdown:

Hello from a docx fixture.



## 5. Closing notes

- **Limits**: at most 4 MiB per attachment, at most 8 attachments per
  run (`ConfigurationError` beyond that). `Attachment(...)` enforces the
  size bound at construction time for `path=`; `Agent.run` enforces the
  count.
- **Scanned PDFs**: a scanned/image-only PDF is not a parse failure. The
  middleware injects a one-line note instead of Markdown: `[Attached
  document "..." is a scanned PDF; text extraction requires OCR, which
  is not enabled.]`. This crate performs no OCR in this release.
- **Fail-soft**: every other failure mode (unreadable source, unsupported
  format, parse error) also collapses to a one-line replacement note; a
  bad attachment never aborts the run.
- **Error codes** (`document_*`, raised as `ValueError` from the debug
  helpers above): `document_invalid_arguments`, `document_parse_failed`,
  `document_too_large`, `document_source_unavailable`,
  `document_unsupported_format`, `document_path_unsupported`.
- **More detail**: see
  [`extensions/toolsets/finstack-ai-tools-document/README.md`](../../extensions/toolsets/finstack-ai-tools-document/README.md)
  and
  [`extensions/middleware/finstack-ai-middleware-document-ingest/README.md`](../../extensions/middleware/finstack-ai-middleware-document-ingest/README.md).